In [1]:
import os
import numpy as np
import pandas as pd

from skimage.io import imread
from skimage.measure import label, regionprops

from tqdm.notebook import tqdm

In [2]:
PROJECT_DIR = r"D:\hiPSC_Morphology_Project"

IMAGE_DIR = os.path.join(PROJECT_DIR, "data", "raw_segmented_images")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "data", "processed")

FEATURES_CSV = os.path.join(OUTPUT_DIR, "features.csv")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Image folder:", IMAGE_DIR)
print("Output folder:", OUTPUT_DIR)

Project folder: D:\hiPSC_Morphology_Project
Image folder: D:\hiPSC_Morphology_Project\data\raw_segmented_images
Output folder: D:\hiPSC_Morphology_Project\data\processed


In [3]:
MIN_AREA = 50

VALID_EXTENSIONS = (
    ".png",
    ".tif",
    ".tiff"
)

In [4]:
def extract_colony_features(image_path):

    image = imread(image_path)

    # Convert RGB to grayscale if necessary
    if image.ndim == 3:
        image = image[:, :, 0]

    # Binary image
    binary = image > 0

    # Label colonies
    labeled = label(binary)

    properties = regionprops(labeled)

    rows = []

    image_name = os.path.basename(image_path)

    for colony_id, prop in enumerate(properties, start=1):

        if prop.area < MIN_AREA:
            continue

        rows.append({

            "image_name": image_name,
            "colony_id": colony_id,

            "area": prop.area,
            "perimeter": prop.perimeter,
            "eccentricity": prop.eccentricity,
            "solidity": prop.solidity,
            "extent": prop.extent,
           "major_axis_length": prop.axis_major_length,
"minor_axis_length": prop.axis_minor_length,
"convex_area": prop.area_convex,
"equivalent_diameter": prop.equivalent_diameter_area
        })

    return pd.DataFrame(rows)

In [5]:
image_paths = sorted([

    os.path.join(IMAGE_DIR, f)

    for f in os.listdir(IMAGE_DIR)

    if f.lower().endswith(VALID_EXTENSIONS)

])

print(f"Found {len(image_paths)} images.")

Found 2890 images.


In [6]:
import os
import pandas as pd

FEATURES_CSV = os.path.join(OUTPUT_DIR, "features.csv")
LOG_CSV = os.path.join(OUTPUT_DIR, "processing_log.csv")

# Create empty files if they don't exist
if not os.path.exists(FEATURES_CSV):
    pd.DataFrame().to_csv(FEATURES_CSV, index=False)

if not os.path.exists(LOG_CSV):
    pd.DataFrame(columns=[
        "image_name",
        "colonies_found",
        "processing_time_seconds",
        "status"
    ]).to_csv(LOG_CSV, index=False)

print("Output files ready.")

Output files ready.


In [7]:
log_df = pd.read_csv(LOG_CSV)

processed_images = set(log_df["image_name"])

print(f"Already processed: {len(processed_images)} images")

Already processed: 0 images


In [8]:
import time

for i, image_path in enumerate(image_paths):

    image_name = os.path.basename(image_path)

    if image_name in processed_images:
        continue

    start = time.time()

    try:

        df = extract_colony_features(image_path)

        # Save extracted features
        if os.path.getsize(FEATURES_CSV) == 0:
            df.to_csv(FEATURES_CSV, index=False)
        else:
            df.to_csv(FEATURES_CSV, mode="a", header=False, index=False)

        elapsed = round(time.time() - start, 2)

        log_entry = pd.DataFrame([{
            "image_name": image_name,
            "colonies_found": len(df),
            "processing_time_seconds": elapsed,
            "status": "Success"
        }])

        log_entry.to_csv(
            LOG_CSV,
            mode="a",
            header=False,
            index=False
        )

        print(
            f"[{i+1}/{len(image_paths)}] "
            f"{image_name} | "
            f"{len(df)} colonies | "
            f"{elapsed:.2f}s"
        )

    except Exception as e:

        elapsed = round(time.time() - start, 2)

        log_entry = pd.DataFrame([{
            "image_name": image_name,
            "colonies_found": -1,
            "processing_time_seconds": elapsed,
            "status": f"Failed: {e}"
        }])

        log_entry.to_csv(
            LOG_CSV,
            mode="a",
            header=False,
            index=False
        )

        print(f"FAILED: {image_name}")

[1/2890] 00031b2f_3500000969_10X_20170613_11_27(10)-Scene-10-F8-F08_seg.tiff | 24 colonies | 1.04s
[2/2890] 001bf60f_3500001005_10X_20170623_11_27(4)-Scene-4-E7-E07_seg.tiff | 48 colonies | 1.07s
[3/2890] 002a7044_3500001388_10X_20171009_23_20-Scene-06-F4-F04_seg.tiff | 26 colonies | 1.92s
[4/2890] 003201bf_3500000778_10X_20170331_11_27(4)-Scene-4-E7-E07_seg.tiff | 29 colonies | 0.55s
[5/2890] 0035d714_3500001319_10X_20170922_24_80-Scene-06-F4-F04_seg.tiff | 12 colonies | 3.09s
[6/2890] 00418fdb_9999999185_10X_20180316_7_79-Scene-07-F5-F05_seg.tiff | 14 colonies | 1.58s
[7/2890] 004e7de5_3500002002_10X_20180504_1-Scene-13-E6-E06_seg.tiff | 0 colonies | 0.05s
[8/2890] 005927a2_3500000876_10X_20170508_13_210-Scene-02-E5-E05_seg.tiff | 21 colonies | 2.24s
[9/2890] 005c6e66_3500003044_10X_20190531_G6_seg.tiff | 38 colonies | 1.81s
[10/2890] 00676311_3500002921_10X_20190419_F3_seg.tiff | 0 colonies | 0.32s
[11/2890] 006cdf33_3500002410_10X_20181012-Scene-03-D4-D04_seg.tiff | 0 colonies | 0.

In [9]:
features_df = pd.read_csv(FEATURES_CSV)
log_df = pd.read_csv(LOG_CSV)

print("\nExtraction Complete!")
print(f"Images processed: {len(log_df)}")
print(f"Total colonies: {len(features_df)}")
print(f"Failed images: {(log_df['status'] != 'Success').sum()}")


Extraction Complete!
Images processed: 2890
Total colonies: 69713
Failed images: 4


In [10]:
features_df = pd.read_csv(FEATURES_CSV)

print("Dataset shape:")
print(features_df.shape)

print("\nDataset info:")
features_df.info()

Dataset shape:
(69713, 11)

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 69713 entries, 0 to 69712
Data columns (total 11 columns):
 #   Column                                                               Non-Null Count  Dtype  
---  ------                                                               --------------  -----  
 0   00031b2f_3500000969_10X_20170613_11_27(10)-Scene-10-F8-F08_seg.tiff  69713 non-null  str    
 1   1                                                                    69713 non-null  int64  
 2   4420.0                                                               69713 non-null  float64
 3   306.8061325481597                                                    69713 non-null  float64
 4   0.8787382954447599                                                   69713 non-null  float64
 5   0.8550976978138906                                                   69713 non-null  float64
 6   0.5328511151295962                                                   69

In [11]:
print(features_df.isnull().sum())

00031b2f_3500000969_10X_20170613_11_27(10)-Scene-10-F8-F08_seg.tiff    0
1                                                                      0
4420.0                                                                 0
306.8061325481597                                                      0
0.8787382954447599                                                     0
0.8550976978138906                                                     0
0.5328511151295962                                                     0
113.10821751146602                                                     0
53.98699276722581                                                      0
5169.0                                                                 0
75.01812306189365                                                      0
dtype: int64


In [12]:
features_df.head(10)

,00031b2f_3500000969_10X_20170613_11_27(10)-Scene-10-F8-F08_seg.tiff,1,4420.0,306.8061325481597,0.8787382954447599,0.8550976978138906,0.5328511151295962,113.10821751146602,53.98699276722581,5169.0,75.01812306189365
0,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,2,1605.0,161.480231,0.778643,0.951393,0.597098,57.766271,36.246447,1687.0,45.205635
1,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,3,1216.0,136.781746,0.271627,0.970471,0.643386,41.052407,39.508946,1253.0,39.347926
2,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,4,549.0,91.112698,0.779800,0.973404,0.769986,33.845157,21.188053,564.0,26.438769
3,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,5,1412.0,146.994949,0.672782,0.957938,0.626442,49.652234,36.734742,1474.0,42.400640
4,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,6,1399.0,182.130988,0.721516,0.858282,0.471520,56.147844,38.876645,1630.0,42.205001
5,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,7,1054.0,140.160426,0.830794,0.868920,0.614577,50.839834,28.296431,1213.0,36.633243
6,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,8,2344.0,252.291414,0.907209,0.743419,0.505064,91.555059,38.515324,3153.0,54.630335
7,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,9,828.0,114.740115,0.598543,0.925140,0.678133,36.839142,29.511500,895.0,32.469098
8,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,10,1205.0,145.095454,0.633495,0.880205,0.640957,45.328385,35.072710,1369.0,39.169550
9,00031b2f_3500000969_10X_20170613_11_27(10)-Sce...,11,647.0,97.982756,0.546834,0.951471,0.652218,31.687828,26.530351,680.0,28.701672


In [13]:
print(len(image_paths))

2890


In [15]:
print(features_df.shape)

(69713, 11)


In [14]:
print("Total colonies:", len(features_df))

Total colonies: 69713


In [16]:
log_df = pd.read_csv(LOG_CSV)

print("Images processed:", len(log_df))
print(log_df["status"].value_counts())

Images processed: 2890
status
Success                                              2886
Failed: failed to read 8377544 bytes, got 786112        1
Failed: not a TIFF file: header=b''                     1
Failed: failed to read 8348256 bytes, got 1572544       1
Failed: failed to read 8376536 bytes, got 1310400       1
Name: count, dtype: int64


In [17]:
features_df.isnull().sum()

00031b2f_3500000969_10X_20170613_11_27(10)-Scene-10-F8-F08_seg.tiff    0
1                                                                      0
4420.0                                                                 0
306.8061325481597                                                      0
0.8787382954447599                                                     0
0.8550976978138906                                                     0
0.5328511151295962                                                     0
113.10821751146602                                                     0
53.98699276722581                                                      0
5169.0                                                                 0
75.01812306189365                                                      0
dtype: int64